# Notebook 08 — Empirical Backtest on LOBSTER Equities (v3)**Corrections from v2** (Prof. Rosenbaum feedback on SPY results):1. **Quotes clamped to half-spread** — no inside-spread quotes allowed2. **Ambiguous intervals skipped** — if both sides touched, skip (not bid-first bias)3. **Terminal liquidation cost** — inventory at end penalised by half-spread4. **Tick discretization** — quotes rounded to $0.01 grid**Carried from v2**: Δ=1 share, independent naive benchmark, rescaled γ grid.**Strategies**: Optimal A (ξ=γ), Optimal B (ξ=0), GLFT closed-form, Naive (median half-spread).

## §0 — Imports and configuration

In [ ]:
import json
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import norm, kurtosis

import sys
sys.path.append(str(Path().resolve().parents[0]))

from market_making.core.solver_1d import solve_general
from market_making.core.closed_form import approx_quotes

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8")
%matplotlib inline

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "data" / "calibrated"
LOBSTER_DIR = ROOT / "data" / "data" / "lobster_raw"

PARAMS_FILE = DATA_DIR / "calibrated_params.json"
assert PARAMS_FILE.exists(), f"Run calibration first — missing {PARAMS_FILE}"

SYMBOLS = ["AAPL", "SPY", "TSLA"]

STRATEGIES = ["optimal", "optimal_b", "naive", "closed_form"]
LABELS = {
    "optimal": "Optimal (Model A)",
    "optimal_b": "Optimal (Model B)",
    "naive": "Naive (symmetric)",
    "closed_form": "Closed-form (GLFT)",
}
COLORS = {
    "optimal": "#00a882",
    "optimal_b": "#f59e0b",
    "naive": "#e05050",
    "closed_form": "#8b5cf6",
}

PRICE_FACTOR = 1e4
SENTINEL_ASK = 9999999999
SENTINEL_BID = -9999999999
RTH_START = 34200
RTH_END = 57600
TICK = 0.01

WINDOW_MINUTES = 30
STRIDE_MINUTES = 15
MAX_WINDOWS = 200
MAKER_FEE = 0.0

print("§0 — Configuration loaded.")

## §1 — Load calibrated parameters (Δ = 1 share)

In [ ]:
with open(PARAMS_FILE, "r") as f:
    raw_params = json.load(f)

all_params = {}
for symbol in SYMBOLS:
    if symbol not in raw_params:
        print(f"WARNING: {symbol} not found in calibrated_params.json")
        continue
    p = raw_params[symbol]
    all_params[symbol] = {
        "sigma": p["sigma"], "A": p["A"], "k": p["k"],
        "Delta": 1.0, "Q": int(p.get("Q", 4)), "lot_size": 1.0,
        "_mean_price": p.get("mean_price", p["Delta"]),
    }

print("Loaded (Delta = 1 share for all):")
for s, p in all_params.items():
    print(f"  {s:<5s} | sigma={p['sigma']:.6f} | A={p['A']:.4f} | "
          f"k={p['k']:.2f} | mean_price={p['_mean_price']:.2f}")

## §2 — Folder discovery

In [ ]:
def parse_folder_metadata(folder_name: str) -> dict:
    parts = folder_name.split("_")
    return {"symbol": parts[0], "start_date": parts[1],
            "end_date": parts[2], "levels": int(parts[3])}

def get_lobster_files(folder: Path):
    msg_files = sorted(folder.glob("*_message_*.csv"))
    ob_files = sorted(folder.glob("*_orderbook_*.csv"))
    if len(msg_files) != 1 or len(ob_files) != 1:
        raise ValueError(f"Expected 1+1 in {folder}, got {len(msg_files)}+{len(ob_files)}")
    return msg_files[0], ob_files[0]

folders = sorted([d for d in LOBSTER_DIR.iterdir() if d.is_dir()]) if LOBSTER_DIR.exists() else []
selected_folders = [f for f in folders
                    if parse_folder_metadata(f.name)["symbol"] in SYMBOLS]

print(f"Selected folders ({len(selected_folders)}):")
for f in selected_folders:
    print(" ", f.name)

## §3 — Parse LOBSTER → interval table

In [ ]:
MSG_COLS_LOB = ["timestamp", "event_type", "order_id", "size", "price", "direction"]

def parse_lobster_to_intervals(folder: Path):
    folder = Path(folder)
    meta = parse_folder_metadata(folder.name)
    symbol, date = meta["symbol"], meta["start_date"]
    msg_file, ob_file = get_lobster_files(folder)

    msg = pd.read_csv(msg_file, header=None, usecols=range(6),
                      names=MSG_COLS_LOB, low_memory=False)
    msg["price"] = msg["price"] / PRICE_FACTOR

    ob = pd.read_csv(ob_file, header=None)
    n_lev = ob.shape[1] // 4
    cols = []
    for i in range(1, n_lev + 1):
        cols += [f"ask{i}_price", f"ask{i}_size", f"bid{i}_price", f"bid{i}_size"]
    ob.columns = cols
    for col in [c for c in ob.columns if "price" in c]:
        ob[col] = ob[col].replace(SENTINEL_ASK, np.nan).replace(SENTINEL_BID, np.nan) / PRICE_FACTOR

    base = pd.Timestamp(date)
    msg["datetime"] = base + pd.to_timedelta(msg["timestamp"], unit="s")

    df = pd.DataFrame({
        "datetime": msg["datetime"].values, "timestamp_s": msg["timestamp"].values,
        "event_type": msg["event_type"].values, "direction": msg["direction"].values,
        "price": msg["price"].values,
        "ask_after": ob["ask1_price"].values, "bid_after": ob["bid1_price"].values,
    })
    df = df[(df["timestamp_s"] >= RTH_START) & (df["timestamp_s"] <= RTH_END)].copy()
    df = df[df["ask_after"].notna() & df["bid_after"].notna() & (df["ask_after"] > df["bid_after"])].copy()

    df["ask_before"] = df["ask_after"].shift(1)
    df["bid_before"] = df["bid_after"].shift(1)
    df = df.dropna(subset=["ask_before", "bid_before"]).copy()
    df = df[df["ask_before"] > df["bid_before"]].copy()

    df["mid"] = (df["ask_before"] + df["bid_before"]) / 2.0
    df["half_spread"] = (df["ask_before"] - df["bid_before"]) / 2.0
    df["is_trade"] = df["event_type"].isin([4, 5])

    l1_changed = ((df["ask_before"] != df["ask_before"].shift(1)) |
                  (df["bid_before"] != df["bid_before"].shift(1)))
    l1_changed.iloc[0] = True
    df["interval_id"] = l1_changed.cumsum() - 1

    lob = df.groupby("interval_id").agg(
        timestamp=("datetime", "first"), mid=("mid", "first"),
        half_spread=("half_spread", "first"),
    )
    trades = df[df["is_trade"]].copy()
    lob["max_buy_price"] = (trades[trades["direction"] == -1]
        .groupby("interval_id")["price"].max().reindex(lob.index))
    lob["min_sell_price"] = (trades[trades["direction"] == 1]
        .groupby("interval_id")["price"].min().reindex(lob.index))

    ts = lob["timestamp"].values
    dt = np.zeros(len(ts))
    if len(ts) > 1:
        dt[:-1] = np.diff(ts.astype("int64")) / 1e9
        dt[-1] = dt[-2]
    elif len(ts) == 1:
        dt[0] = 1.0
    lob["dt"] = dt
    lob = lob[lob["dt"] > 0].reset_index(drop=True)
    return lob, symbol

## §4 — Load all interval tables

In [ ]:
all_intervals = {}
per_day = {}
for folder in selected_folders:
    try:
        iv, sym = parse_lobster_to_intervals(folder)
        per_day[f"{sym}_{parse_folder_metadata(folder.name)['start_date']}"] = (sym, iv)
    except Exception as e:
        print(f"FAILED on {folder.name}: {e}")

for symbol in SYMBOLS:
    sym_ivs = [iv for _, (s, iv) in per_day.items() if s == symbol]
    if sym_ivs:
        all_intervals[symbol] = pd.concat(sym_ivs, ignore_index=True)

print(f"{'Symbol':<8s} {'σ':>10s} {'A':>10s} {'k':>10s} {'Intervals':>12s}")
print(f"{'-'*50}")
for sym in SYMBOLS:
    if sym not in all_intervals or sym not in all_params:
        continue
    p = all_params[sym]
    print(f"{sym:<8s} {p['sigma']:>10.6f} {p['A']:>10.4f} "
          f"{p['k']:>10.2f} {len(all_intervals[sym]):>12,}")

## §5 — Diagnostics

In [ ]:
for symbol in SYMBOLS:
    if symbol not in all_intervals:
        continue
    iv = all_intervals[symbol]
    p = all_params[symbol]
    sigma = p["sigma"]

    iv_ts = iv.set_index("timestamp")
    unique_sessions = sorted(set(iv_ts.index.date))
    all_ret = []
    for sess in unique_sessions:
        mid_sess = iv_ts.loc[iv_ts.index.date == sess, "mid"]
        mid_1s = mid_sess.resample("1s").last().dropna()
        all_ret.append(mid_1s.diff().dropna())
    returns_1s = pd.concat(all_ret) if all_ret else pd.Series(dtype=float)
    returns_nz = returns_1s[returns_1s != 0]

    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
    fig.suptitle(f"{symbol} — Diagnostics (σ={sigma:.4f}, A={p['A']:.2f}, k={p['k']:.1f})",
                 fontsize=14, fontweight="bold")

    ax = axes[0]
    mid_1m = iv_ts["mid"].resample("1min").last().dropna()
    ax.plot(mid_1m.index, mid_1m.values, lw=0.4, color="steelblue")
    ax.set_title("Mid-price (1min)"); ax.set_ylabel("$"); ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=7)

    ax = axes[1]
    if len(returns_nz) > 50:
        q_lo, q_hi = returns_nz.quantile([0.01, 0.99])
        trim = returns_nz[(returns_nz > q_lo) & (returns_nz < q_hi)]
        ax.hist(trim, bins=100, density=True, alpha=0.6, color="steelblue", label="Empirical")
        x_g = np.linspace(q_lo, q_hi, 300)
        ax.plot(x_g, norm.pdf(x_g, 0, sigma), "r-", lw=2, label="N(0,σ²)")
        ax.legend(fontsize=7)
    ax.set_title(f"1s Returns (kurt={kurtosis(returns_nz):.1f})" if len(returns_nz) > 50 else "1s Returns")
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    hs = iv["half_spread"].dropna(); cap = hs.quantile(0.99)
    ax.hist(hs[hs <= cap], bins=80, density=True, alpha=0.7, color="darkorange")
    ax.axvline(hs.median(), color="red", ls="--", lw=2, label=f"med={hs.median():.4f}")
    ax.set_title("Half-spread"); ax.set_xlabel("$"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    ax = axes[3]
    has_trades = iv["max_buy_price"].notna() | iv["min_sell_price"].notna()
    trade_iv = iv[has_trades].set_index("timestamp")
    if len(trade_iv) > 0:
        rate = trade_iv.resample("1min").size()
        ax.plot(rate.index, rate.values, lw=0.4, color="darkorange")
        ax.set_title(f"Trades/min (mean={rate.mean():.0f})")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.tight_layout(); plt.show()

print("§5 — Diagnostics complete.")

## §6 — Backtest engine (v3)**All 4 fixes from prof feedback**:1. **Quotes clamped to half-spread**: `db_eff = max(db, hs_i)` — prevents inside-spread quotes2. **Ambiguous intervals skipped**: if both bid and ask are touched, no fill (instead of bid-first bias)3. **Terminal liquidation cost**: `mtm -= |q_T| × lot_size × terminal_half_spread`4. **Tick discretization**: bid floored, ask ceiled to $0.01 grid

In [ ]:
@dataclass
class EmpiricalBtResult:
    timestamps: np.ndarray
    mid_prices: np.ndarray
    inventory: np.ndarray
    cash: np.ndarray
    mtm: np.ndarray
    n_bid_fills: int
    n_ask_fills: int
    n_bid_passive: int
    n_ask_passive: int
    delta_bid_posted: np.ndarray
    delta_ask_posted: np.ndarray
    market_half_spread: np.ndarray
    fees_paid: float
    n_ambiguous_skipped: int
    strategy: str
    gamma: float
    symbol: str

    @property
    def pnl(self):
        return float(self.mtm[-1])

    @property
    def total_fills(self):
        return self.n_bid_fills + self.n_ask_fills

    @property
    def final_inventory(self):
        return int(self.inventory[-1])

    @property
    def mean_spread(self):
        return float(np.nanmean(self.delta_bid_posted + self.delta_ask_posted))

In [ ]:
def run_backtest_real_lob(intervals, params, gamma, strategy="optimal",
                          maker_fee=0.0, symbol=""):
    Delta = params["Delta"]
    Q = int(params["Q"])
    lot_size = params["lot_size"]

    intervals = intervals.sort_values("timestamp").reset_index(drop=True)
    N = len(intervals)
    T_total = float(intervals["dt"].sum())
    N_t_ode = max(300, int(T_total))

    sol_a = solve_general(params, gamma, T_total, xi=gamma, N_t=N_t_ode)
    sol_b = solve_general(params, gamma, T_total, xi=0.0, N_t=N_t_ode)

    if strategy == "optimal_b":
        db_table, da_table = sol_b["delta_bid"], sol_b["delta_ask"]
    else:
        db_table, da_table = sol_a["delta_bid"], sol_a["delta_ask"]

    # Independent naive benchmark = median market half-spread
    half_spread_naive = float(np.median(intervals["half_spread"].dropna()))

    db_cf = da_cf = None
    if strategy == "closed_form":
        db_cf, da_cf = approx_quotes(np.arange(-Q, Q + 1), params, gamma, xi=gamma)

    mids = intervals["mid"].values
    dts = intervals["dt"].values
    hs = intervals["half_spread"].values
    mbuy = intervals["max_buy_price"].values
    msell = intervals["min_sell_price"].values
    ts = intervals["timestamp"].values

    inv_arr = np.zeros(N + 1, dtype=int)
    cash_arr = np.zeros(N + 1)
    mtm_arr = np.zeros(N + 1)
    db_post = np.full(N, np.nan)
    da_post = np.full(N, np.nan)

    n_inv = 0
    X = 0.0
    nb = na = nbp = nap = n_ambig = 0
    fees = 0.0
    cum_t = np.insert(np.cumsum(dts), 0, 0.0)

    for i in range(N):
        mid_i = mids[i]
        hs_i = hs[i]
        i_lot = n_inv + Q

        # ── Quote selection ──
        if strategy in ("optimal", "optimal_b"):
            t_idx = min(int(cum_t[i] / T_total * N_t_ode), N_t_ode - 1) if T_total > 0 else 0
            db = db_table[t_idx, i_lot] if n_inv < Q and np.isfinite(db_table[t_idx, i_lot]) else np.inf
            da = da_table[t_idx, i_lot] if n_inv > -Q and np.isfinite(da_table[t_idx, i_lot]) else np.inf
        elif strategy == "naive":
            db = half_spread_naive if n_inv < Q else np.inf
            da = half_spread_naive if n_inv > -Q else np.inf
        elif strategy == "closed_form":
            db = db_cf[i_lot] if n_inv < Q and 0 <= i_lot < len(db_cf) else np.inf
            da = da_cf[i_lot] if n_inv > -Q and 0 <= i_lot < len(da_cf) else np.inf
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        # ── FIX #1: Clamp quotes to at least the half-spread ──
        # Prevents inside-spread quotes (non-passive)
        if np.isfinite(db) and not np.isnan(hs_i):
            db_eff = max(db, hs_i)
        else:
            db_eff = db
        if np.isfinite(da) and not np.isnan(hs_i):
            da_eff = max(da, hs_i)
        else:
            da_eff = da

        bp = np.isfinite(db_eff) and db_eff >= 0
        ap = np.isfinite(da_eff) and da_eff >= 0

        # ── FIX #4: Tick discretization ──
        if bp:
            bid_p = np.floor((mid_i - db_eff) / TICK) * TICK
            db_eff = mid_i - bid_p   # recalculate effective distance
        else:
            bid_p = np.nan

        if ap:
            ask_p = np.ceil((mid_i + da_eff) / TICK) * TICK
            da_eff = ask_p - mid_i
        else:
            ask_p = np.nan

        db_post[i] = db_eff
        da_post[i] = da_eff

        if bp:
            nbp += 1
        if ap:
            nap += 1

        # ── Check fills ──
        bid_hit = bp and np.isfinite(msell[i]) and msell[i] <= bid_p
        ask_hit = ap and np.isfinite(mbuy[i]) and mbuy[i] >= ask_p

        # ── FIX #2: Skip ambiguous intervals ──
        if bid_hit and ask_hit:
            n_ambig += 1
            # Don't fill either side — we don't know the order
        elif bid_hit:
            fee = maker_fee * abs(bid_p) * lot_size
            X -= bid_p * lot_size + fee
            n_inv = min(n_inv + 1, Q)
            nb += 1
            fees += fee
        elif ask_hit:
            fee = maker_fee * abs(ask_p) * lot_size
            X += ask_p * lot_size - fee
            n_inv = max(n_inv - 1, -Q)
            na += 1
            fees += fee

        inv_arr[i + 1] = n_inv
        cash_arr[i + 1] = X
        mtm_arr[i + 1] = X + n_inv * lot_size * mid_i

    # ── FIX #3: Terminal liquidation cost ──
    valid_hs = hs[~np.isnan(hs)]
    terminal_hs = valid_hs[-1] if len(valid_hs) > 0 else 0.0
    q_T = inv_arr[-1]
    mtm_arr[-1] = (cash_arr[-1]
                    + q_T * lot_size * mids[-1]
                    - abs(q_T) * lot_size * terminal_hs)

    return EmpiricalBtResult(
        timestamps=ts, mid_prices=mids, inventory=inv_arr,
        cash=cash_arr, mtm=mtm_arr,
        n_bid_fills=nb, n_ask_fills=na,
        n_bid_passive=nbp, n_ask_passive=nap,
        delta_bid_posted=db_post, delta_ask_posted=da_post,
        market_half_spread=hs.copy(), fees_paid=fees,
        n_ambiguous_skipped=n_ambig,
        strategy=strategy, gamma=gamma, symbol=symbol,
    )

print("§6 — Backtest engine v3 defined.")
print("  Fixes: clamp-to-spread, skip-ambiguous, terminal-liquidation, tick-discretize")

## §7 — Gamma sweep

In [ ]:
GAMMA_CANDIDATES = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05]
SWEEP_WINDOWS = 20
best_gamma = {}

for symbol in SYMBOLS:
    if symbol not in all_intervals:
        continue
    iv = all_intervals[symbol].copy()
    params = all_params[symbol]
    iv["_session"] = iv["timestamp"].dt.date
    sessions = sorted(iv["_session"].unique())
    sweep_results = {g: [] for g in GAMMA_CANDIDATES}

    n_done = 0
    for sess in sessions:
        if n_done >= SWEEP_WINDOWS:
            break
        sess_iv = iv[iv["_session"] == sess].drop(columns=["_session"]).copy()
        if len(sess_iv) < 50:
            continue
        mid_ts = sess_iv["timestamp"].iloc[len(sess_iv) // 2]
        w = sess_iv[(sess_iv["timestamp"] >= mid_ts) &
                    (sess_iv["timestamp"] < mid_ts + pd.Timedelta(minutes=30))].copy()
        if len(w) < 50:
            continue
        for g in GAMMA_CANDIDATES:
            try:
                res = run_backtest_real_lob(w, params, g, strategy="optimal",
                                            maker_fee=MAKER_FEE, symbol=symbol)
                sweep_results[g].append(res.pnl)
            except Exception:
                pass
        n_done += 1
    iv.drop(columns=["_session"], inplace=True, errors="ignore")

    best_sr, best_g = -np.inf, 0.001
    print(f"\n{symbol} γ sweep ({n_done} windows):")
    for g in GAMMA_CANDIDATES:
        pnls = np.array(sweep_results[g])
        if len(pnls) < 3:
            continue
        sr = pnls.mean() / pnls.std() if pnls.std() > 0 else 0.0
        print(f"  γ={g:<8.4f}  mean={pnls.mean():+.4f}  std={pnls.std():.4f}  Sharpe={sr:.3f}")
        if sr > best_sr:
            best_sr, best_g = sr, g
    best_gamma[symbol] = best_g
    print(f"  → Best γ = {best_g}")

print(f"\nSelected γ:")
for sym, g in best_gamma.items():
    print(f"  {sym}: {g}")

## §8 — Rolling backtest

In [ ]:
rolling_all = {}

for symbol in SYMBOLS:
    if symbol not in all_intervals:
        continue
    iv = all_intervals[symbol].copy()
    params = all_params[symbol]
    gamma = best_gamma.get(symbol, 0.001)

    print(f"\n{'='*70}")
    print(f"{symbol}  γ={gamma}  Δ={params['Delta']:.1f}")
    print(f"{'='*70}")

    iv["_session"] = iv["timestamp"].dt.date
    sessions = sorted(iv["_session"].unique())
    results_list = []
    win_count = 0

    for sess in sessions:
        sess_iv = iv[iv["_session"] == sess].drop(columns=["_session"]).copy()
        starts = pd.date_range(
            sess_iv["timestamp"].min() + pd.Timedelta(minutes=5),
            sess_iv["timestamp"].max() - pd.Timedelta(minutes=WINDOW_MINUTES + 5),
            freq=f"{STRIDE_MINUTES}min")
        for start in starts:
            if win_count >= MAX_WINDOWS:
                break
            w = sess_iv[(sess_iv["timestamp"] >= start) &
                        (sess_iv["timestamp"] < start + pd.Timedelta(minutes=WINDOW_MINUTES))].copy()
            if len(w) < 50:
                continue
            row = {
                "start": start, "session": str(sess), "n_intervals": len(w),
                "mid_price": float(w["mid"].mean()),
                "sigma_realised": float(np.std(np.diff(w["mid"].values))) if len(w) > 1 else 0.0,
                "mean_half_spread": float(w["half_spread"].mean()),
            }
            for strat in STRATEGIES:
                try:
                    res = run_backtest_real_lob(w, params, gamma, strategy=strat,
                                                maker_fee=MAKER_FEE, symbol=symbol)
                    row[f"{strat}_pnl"] = res.pnl
                    row[f"{strat}_fills"] = res.total_fills
                    row[f"{strat}_inv_T"] = res.final_inventory
                    row[f"{strat}_abs_inv_mean"] = float(np.mean(np.abs(res.inventory)))
                    row[f"{strat}_ambiguous"] = res.n_ambiguous_skipped
                except Exception:
                    row[f"{strat}_pnl"] = np.nan
            results_list.append(row)
            win_count += 1
            if win_count % 50 == 0:
                print(f"  [{win_count}/{MAX_WINDOWS}]")
        if win_count >= MAX_WINDOWS:
            break
    iv.drop(columns=["_session"], inplace=True, errors="ignore")
    roll_df = pd.DataFrame(results_list)
    rolling_all[symbol] = roll_df
    print(f"  → {len(roll_df)} windows")

    # Quick ambiguity diagnostic
    if "optimal_ambiguous" in roll_df.columns:
        pct = roll_df["optimal_ambiguous"].sum() / roll_df["n_intervals"].sum() * 100
        print(f"  Ambiguous intervals skipped: {pct:.1f}%")

print("\n§8 complete.")

## §9 — Per-symbol diagnostics

In [ ]:
for symbol in SYMBOLS:
    if symbol not in rolling_all:
        continue
    roll_df = rolling_all[symbol]
    gamma = best_gamma.get(symbol, 0.001)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    ax = axes[0, 0]
    box_data = [roll_df[f"{s}_pnl"].dropna().values for s in STRATEGIES if f"{s}_pnl" in roll_df]
    box_lbls = [LABELS[s] for s in STRATEGIES if f"{s}_pnl" in roll_df]
    bp = ax.boxplot(box_data, labels=box_lbls, patch_artist=True, widths=0.6)
    for patch, s in zip(bp["boxes"], [s for s in STRATEGIES if f"{s}_pnl" in roll_df]):
        patch.set_facecolor(COLORS[s]); patch.set_alpha(0.5)
    ax.set_title("P&L Distribution"); ax.set_ylabel("P&L ($)"); ax.grid(True, alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right", fontsize=8)

    ax = axes[0, 1]
    for strat in STRATEGIES:
        col = f"{strat}_pnl"
        if col in roll_df:
            ax.plot(range(len(roll_df)), roll_df[col].fillna(0).cumsum(),
                    lw=2, color=COLORS[strat], label=LABELS[strat])
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title("Cumulative P&L"); ax.set_xlabel("Window #"); ax.set_ylabel("$")
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    for strat in STRATEGIES:
        col = f"{strat}_pnl"
        if col in roll_df:
            ax.scatter(roll_df["sigma_realised"], roll_df[col],
                       s=10, alpha=0.4, color=COLORS[strat], label=LABELS[strat])
    ax.set_title("P&L vs Realised σ"); ax.set_xlabel("σ realised"); ax.set_ylabel("P&L ($)")
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    others = [s for s in STRATEGIES if s != "optimal"]
    wr = []
    for o in others:
        v = roll_df[["optimal_pnl", f"{o}_pnl"]].dropna()
        wr.append((v["optimal_pnl"] > v[f"{o}_pnl"]).mean() * 100 if len(v) else 50)
    ax.bar([f"vs {LABELS[o]}" for o in others], wr,
           color=[COLORS[o] for o in others], alpha=0.7)
    ax.axhline(50, color="k", ls="--", lw=1.5)
    ax.set_title("Optimal win rate"); ax.set_ylabel("% windows"); ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right", fontsize=8)

    plt.suptitle(f"{symbol} — v3 Backtest ({len(roll_df)} win, γ={gamma}, Δ=1)",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout(); plt.show()

## §10 — Cross-symbol summary

In [ ]:
print(f"{'Symbol':<8s} {'γ':>8s} {'Strategy':<22s} {'Mean':>10s} {'Std':>10s} "
      f"{'Sharpe':>8s} {'Opt%':>6s} {'N':>5s}")
print(f"{'-'*82}")
summary_rows = []
for symbol in SYMBOLS:
    if symbol not in rolling_all:
        continue
    roll_df = rolling_all[symbol]
    gamma = best_gamma.get(symbol, 0.001)
    for strat in STRATEGIES:
        col = f"{strat}_pnl"
        if col not in roll_df: continue
        pnls = roll_df[col].dropna()
        std = pnls.std()
        sr = pnls.mean() / std if std > 0 else 0.0
        if strat == "optimal":
            win_str = "—"
        else:
            v = roll_df[["optimal_pnl", col]].dropna()
            win_str = f"{(v['optimal_pnl'] > v[col]).mean() * 100:.0f}%" if len(v) else "?"
        print(f"{symbol:<8s} {gamma:>8.4f} {LABELS[strat]:<22s} "
              f"{pnls.mean():>+10.4f} {std:>10.4f} {sr:>8.3f} {win_str:>6s} {len(pnls):>5d}")
        summary_rows.append({"Symbol": symbol, "gamma": gamma, "Strategy": LABELS[strat],
                             "Mean_PnL": pnls.mean(), "Std_PnL": std, "Sharpe": sr, "N": len(pnls)})
    print()
summary_df = pd.DataFrame(summary_rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
syms_in = [s for s in SYMBOLS if s in rolling_all]
x = np.arange(len(syms_in)); width = 0.2

ax = axes[0]
for i, strat in enumerate(STRATEGIES):
    vals = [summary_df[(summary_df["Symbol"]==sym) & (summary_df["Strategy"]==LABELS[strat])]["Sharpe"].values[0]
            if len(summary_df[(summary_df["Symbol"]==sym) & (summary_df["Strategy"]==LABELS[strat])]) else 0.0
            for sym in syms_in]
    ax.bar(x + i*width, vals, width, color=COLORS[strat], label=LABELS[strat], alpha=0.8)
ax.set_xticks(x + 1.5*width); ax.set_xticklabels(syms_in)
ax.set_title("Sharpe by Symbol × Strategy", fontsize=13, fontweight="bold")
ax.set_ylabel("Sharpe"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")
ax.axhline(0, color="k", lw=0.5)

ax = axes[1]
win_rates = []
for sym in syms_in:
    v = rolling_all[sym][["optimal_pnl", "naive_pnl"]].dropna()
    win_rates.append((v["optimal_pnl"] > v["naive_pnl"]).mean() * 100 if len(v) else 50)
bars = ax.bar(syms_in, win_rates,
              color=["#00a882" if w > 50 else "#e05050" for w in win_rates], alpha=0.8)
ax.axhline(50, color="k", ls="--", lw=1.5)
ax.set_title("Optimal vs Naive: Win Rate", fontsize=13, fontweight="bold")
ax.set_ylabel("% windows"); ax.set_ylim(0, 100); ax.grid(True, alpha=0.3)
for bar, wr in zip(bars, win_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+1.5,
            f"{wr:.0f}%", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()

## §11 — Methodological note (v3)**v3 corrections** (on top of v2):1. **Quote clamping**: `db_eff = max(db, hs_i)` ensures all posted quotes are   at or outside the observed best bid/ask. Without this, the solver could produce   quotes inside the spread — these are not passive limit orders and would   behave as marketable orders in practice.2. **Ambiguous interval handling**: When both bid and ask are touched in the same   L1 regime, we skip the interval entirely. The previous "bid-first" rule introduced   a systematic bias favouring bid-fill strategies. Since we only have trade extremes   (not chronological order), the honest approach is to discard these ambiguous cases.3. **Terminal liquidation cost**: Final mark-to-market now includes   `- |q_T| × lot_size × terminal_half_spread`. This prevents strategies that   accumulate inventory from appearing profitable when they would face real   liquidation costs.4. **Tick discretization**: Bid prices are floored and ask prices are ceiled to the   $0.01 tick grid. This makes the backtest consistent with the NASDAQ tick structure   and prevents unrealistically precise quote placement.**Remaining limitations**:- Fill rule is still reduced-form (price-reached), not queue-aware- Interval compression loses intra-regime event ordering- Model A ≈ Model B is expected when γΔ/k is small (not a bug)- All-negative P&L can reflect adverse selection in the fill rule